In [ ]:
# Parallel Agents in Google ADK: Concurrent Task Execution
# Welcome! In this notebook, you'll learn how to use Parallel Agents in Google's Agent Development Kit (ADK).
# We will build a travel assistant that needs to perform two tasks simultaneously for a user's query, one that
# finds the latest events or news related to a destination/event using the Tavily web search tool, and another that
# creates a detailed itinerary using Google Search for up-to-date recommendations.

In [2]:
# Install Google ADK for Python
%pip install google-adk --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Install LangChain community and Tavily client for the search tool
%pip install langchain_community tavily-python --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# (optional) Verify installation
%pip show google-adk
%pip show langchain_community 
%pip show tavily-python

Name: google-adk
Version: 1.0.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: authlib, click, fastapi, google-api-python-client, google-cloud-aiplatform, google-cloud-secret-manager, google-cloud-speech, google-cloud-storage, google-genai, graphviz, mcp, opentelemetry-api, opentelemetry-exporter-gcp-trace, opentelemetry-sdk, pydantic, python-dotenv, PyYAML, sqlalchemy, tzlocal, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.
Name: langchain-community
Version: 0.3.24
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, reque

In [5]:
# Configure environment
import os
# We use Gemini as our language model and ensure we are not using Vertex AI for this demo.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

In [25]:
# Import Required Modules
from google.adk.agents import Agent, ParallelAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types
from IPython.display import Markdown, display

In [7]:
# Tavily Search API: Set Up
# API Key Management: Always use environment variables or a secure vault for credentials!
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError(
        "TAVILY_API_KEY environment variable not set. "
        "Never hardcode API keys in your code. "
        "Set it securely in your environment or use a secrets manager."
)

In [28]:
# Tavily LangChain tool: Initilaization
# This tool allows the agent to perform live web searches using Tavily.
from langchain_community.tools import TavilySearchResults
tavily_tool_instance = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    include_answer=True,
    include_raw_content=True,
    include_images=True,
)

# Wrap it for ADK compatibility
from google.adk.tools.langchain_tool import LangchainTool
adk_tavily_tool = LangchainTool(tool=tavily_tool_instance)

In [29]:
# Latest Events Agent: Definition
# Define the agent with Tavily as a third-party tool
# The instruction set is the heart of the agent. Make it specific, structured, and safe.
latest_events_agent = Agent(
    name="latest_events_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="An agent that uses the Tavily search tool to find and summarize the latest events, festivals, and activities for a given destination, providing up-to-date and user-friendly travelinformation.",
    instruction="""
        You are a travel assistant.
        When a user asks about current or upcoming events, festivals, or activities at a destination, use the Tavily search tool to find the most recent and relevant information.
        Summarize your findings in a clear, concise, and user-friendly way.
        Only use the Tavily tool for event or activity-related queries, and ensure your answers are accurate and helpful.
        If you cannot find relevant information, politely let the user know.
        Always provide the source of your information.
        If the user asks about general travel advice or information not related to current events, politely redirect them to other resources.
        Never expose API keys or sensitive data in your responses.
    """,
    tools=[adk_tavily_tool],
    output_key="latest_events"
)

In [30]:
# Itinerary Agent: Definition
# This agent creates a detailed itinerary using Google Search for up-to-date recommendations.
itinerary_agent = Agent(
    name="itinerary_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="An agent that creates a travel itinerary for a given destination and trip duration, using Google Search for up-to-date recommendations.",
    instruction="""
        You are a helpful and creative travel itinerary planner.
        When the user provides a destination and trip duration, use the [google_search] tool to find current and popular attractions, restaurants, and activities for each day.
        For each day, create a detailed schedule with 2-4 activities, including at least one meal suggestion and one local attraction, prioritizing the user's interests (such as art and food).
        Present the itinerary in a clear, easy-to-read markdown format, organized by day.
        If you need more information, ask the user for preferences or interests, but if you have enough, proceed to generate a full itinerary.
        Example format:

        ### Day 1
        - Morning: [Attraction or activity]
        - Lunch: [Restaurant or food experience]
        - Afternoon: [Attraction or activity]
        - Evening: [Dinner suggestion or event]

        ### Day 2
        ...

        Always use the [google_search] tool to find the latest recommendations for each activity or restaurant.
        Be concise, friendly, and ensure the itinerary is complete for all days requested.
    """,
    tools=[google_search],
    output_key="itinerary"
)

In [31]:
# Personalizer Agent: Definition
# This agent refines the itinerary by integrating relevant events from the events agent.
personalizer_agent = Agent(
    name="personalizer_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Refines a travel itinerary by integrating relevant, timely events based on user interests.",
    instruction="""
        You are a travel personalization expert. Your task is to review a generated itinerary and a list of current events, and then create a final, unified plan.

        You will receive a pre-generated itinerary and a list of events. Your goal is to see if any of the events from the list can be integrated into the itinerary to make it more personalized and exciting.

        **Instructions:**
        1.  Analyze the itinerary and the list of events.
        2.  Identify events that align with the user's interests (like art, food, music) mentioned in the itinerary.
        3.  If a relevant event's dates overlap with the trip dates, suggest integrating it into the itinerary. You can replace a more generic activity with the specific event.
        4.  Present a single, final, polished itinerary. If you make a change, briefly mention why (e.g., "I've replaced the museum visit with the special 'Shiko Munakata' exhibition, as it aligns with your interest in art.").
        5.  If no events can be reasonably integrated, simply present the original itinerary as the final plan.

        **Input Itinerary:**
        {itinerary}

        **Input Events List:**
        {latest_events}
    """,
    output_key="personalized_itinerary"
)

In [32]:
# Create a parallel agent to run the first two tasks concurrently
gather_info_agent = ParallelAgent(
    name="gather_info_agent",
    sub_agents=[latest_events_agent, itinerary_agent]
)

# Create the final sequential pipeline
travel_pipeline_agent = SequentialAgent(
    name="travel_pipeline_agent",
    sub_agents=[gather_info_agent, personalizer_agent],
    description="A sequential agent that first gathers travel information in parallel, then personalizes the result."
)

In [36]:
# Set up the session and user input
APP_NAME = "wanderwise_workflow_demo"
USER_ID = "user_001"
SESSION_ID = "workflow_session_001"
USER_INPUT = "I'm traveling to Paris for 3 days and I love fashion and food."

# Run the travel planning workflow
session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

Session(id='workflow_session_001', app_name='wanderwise_workflow_demo', user_id='user_001', state={}, events=[], last_update_time=1748839087.1278262)

In [37]:
# Run the full pipeline agent with user input
content = types.Content(role="user", parts=[types.Part(text=USER_INPUT)])
runner = Runner(
    agent=travel_pipeline_agent,
    app_name=APP_NAME,
    session_service=session_service
)
events = runner.run(
    user_id=USER_ID,
    session_id=SESSION_ID,
    new_message=content
)

# Parse and Display Outputs
for event in events:
    if event.is_final_response():
        # Concatenate all parts in the final response
        full_text = ''.join(part.text for part in event.content.parts if part.text)
        display(Markdown(f"{full_text}"))

Okay, I will create a 3-day itinerary for your trip to Paris, focusing on fashion and food.

Here's your 3-day Paris itinerary focusing on fashion and food:

### Day 1: Fashion History & Parisian Classics

*   **Morning:** Start your day with a visit to the **Palais Galliera**, the fashion museum of Paris, showcasing haute couture creations and the evolution of Parisian style. Check out the "Haute Couture: 100 Years of Elegance" exhibition if it's running (typically May-December).
*   **Lunch:** Enjoy a classic Parisian cafe experience at **Café de Flore** in Saint-Germain-des-Prés. Try their famous hot chocolate and a croissant.
*   **Afternoon:** Explore the **Musée Yves Saint Laurent**, dedicated to the life and work of the iconic designer.
*   **Evening:** Indulge in dinner at **Bistrot Paul Bert**, known for its traditional bistro fare and excellent steak frites.

### Day 2: Haute Couture & Modern Trends

*   **Morning:** Visit the **Louvre Museum** and see the "Louvre Couture: Art Objects, Fashion Objects" exhibition (until July 21, 2025), which explores the relationship between fashion and art.
*   **Lunch:** Have lunch at **Restaurant Monsieur Dior**, known for its stylish ambiance and refined cuisine.
*   **Afternoon:** Immerse yourself in the world of luxury at **Le Bon Marché**, a high-end department store.
*   **Evening:** Experience fine dining at **Le Clarence**, a Michelin-starred restaurant offering exquisite French cuisine.

### Day 3: Designer Exploration & Café Culture

*   **Morning:** Explore the **Fondation Azzedine Alaïa**, and if available, view the Azzedine Alaïa and Thierry Mugler exhibition, showcasing the designers' unique relationship and creations.
*   **Lunch:** Enjoy a delightful lunch at **Carette**, known for its macarons and elegant atmosphere, with a view of the Eiffel Tower.
*   **Afternoon:** Stroll along the **Champs-Élysées**, window shopping at flagship designer stores. Alternatively, visit the **Grand Palais** to see the "Dolce&Gabbana: From Heart to Hand" exhibition (until March 31, 2025) for an immersive fashion experience.
*   **Evening:** Conclude your trip with dinner at **Arpège**, a restaurant celebrated for its vegetable-focused haute cuisine and innovative dishes.

Enjoy your fashionable and delicious trip to Paris!


For your trip to Paris in June 2025, here's what I found regarding fashion and food events:

**Fashion Events:**

*   **Paris Fashion Week Runway Event by The Musa Showroom - Mens SS 26:** Slated for June 27, 2025, at EST Galerie, Paris.
*   **DESIGNERS Show your designs in The Stylist Collective - Paris Fashion Week:** Happening on June 27, 2025, at 76 Rue Saint-Maur, Paris.
*   **Le Mazette Discosecte Lma-G Olivier Shiva Viv Yamajoy:** On June 6, 2025, at Le Mazette, Paris.

**Food Events:**

*   **Taste of Paris:** This prestigious food festival is scheduled for May 8 - May 11, 2025, at the Grand Palais Éphémère. It brings together top chefs to showcase French cuisine.
*   **Fête du Pain (The French Bread Festival):** Takes place May 8-18, 2025, in front of Notre Dame Cathedral.

Please note that dates and locations may be subject to change, and it's always a good idea to check the official event websites for the most up-to-date information.

**Sources:**

*   [Allevents.in](https://allevents.in/)
*   [FashionWeekOnline.com](https://fashionweekonline.com/)
*   [visitparisregion.com](https://www.visitparisregion.com/)
*   [katiedonnellyphotography.com](https://katiedonnellyphotography.com/)
*   [parisjetaime.com](https://parisjetaime.com/)
*   [cooknwithclass.com](https://cooknwithclass.com/)

Okay, I've reviewed your itinerary and the list of events in Paris in June 2025. Unfortunately, the identified events don't align well with the provided itinerary. Therefore, the original itinerary will remain unchanged.

Here's your 3-day Paris itinerary focusing on fashion and food:

### Day 1: Fashion History & Parisian Classics

*   **Morning:** Start your day with a visit to the **Palais Galliera**, the fashion museum of Paris, showcasing haute couture creations and the evolution of Parisian style. Check out the "Haute Couture: 100 Years of Elegance" exhibition if it's running (typically May-December).
*   **Lunch:** Enjoy a classic Parisian cafe experience at **Café de Flore** in Saint-Germain-des-Prés. Try their famous hot chocolate and a croissant.
*   **Afternoon:** Explore the **Musée Yves Saint Laurent**, dedicated to the life and work of the iconic designer.
*   **Evening:** Indulge in dinner at **Bistrot Paul Bert**, known for its traditional bistro fare and excellent steak frites.

### Day 2: Haute Couture & Modern Trends

*   **Morning:** Visit the **Louvre Museum** and see the "Louvre Couture: Art Objects, Fashion Objects" exhibition (until July 21, 2025), which explores the relationship between fashion and art.
*   **Lunch:** Have lunch at **Restaurant Monsieur Dior**, known for its stylish ambiance and refined cuisine.
*   **Afternoon:** Immerse yourself in the world of luxury at **Le Bon Marché**, a high-end department store.
*   **Evening:** Experience fine dining at **Le Clarence**, a Michelin-starred restaurant offering exquisite French cuisine.

### Day 3: Designer Exploration & Café Culture

*   **Morning:** Explore the **Fondation Azzedine Alaïa**, and if available, view the Azzedine Alaïa and Thierry Mugler exhibition, showcasing the designers' unique relationship and creations.
*   **Lunch:** Enjoy a delightful lunch at **Carette**, known for its macarons and elegant atmosphere, with a view of the Eiffel Tower.
*   **Afternoon:** Stroll along the **Champs-Élysées**, window shopping at flagship designer stores. Alternatively, visit the **Grand Palais** to see the "Dolce&Gabbana: From Heart to Hand" exhibition (until March 31, 2025) for an immersive fashion experience.
*   **Evening:** Conclude your trip with dinner at **Arpège**, a restaurant celebrated for its vegetable-focused haute cuisine and innovative dishes.

Enjoy your fashionable and delicious trip to Paris!


In [ ]:
# 🎉 Congratulations!
# You've now successfully built and run a ParallelAgent that orchestrates these sub-agents to run their tasks concurrently.